# NOVA - fine-tune YOLOv8n on IDD

Produces `best.pt`: the detector that makes "trained on Indian roads" a
checkable claim rather than a slogan. It drops into NOVA with no code change,
because the class NAMES are the only contract between YOLO and the pipeline.

**Before running this**, on your own laptop (already done if the folder
`C:\NOVA\idd_yolo` exists):

```powershell
python scripts\idd_to_yolo.py --src "C:\Users\Admin\Downloads\idd-detection\IDD_Detection" --dst "C:\NOVA\idd_yolo"
Compress-Archive -Path C:\NOVA\idd_yolo -DestinationPath C:\NOVA\idd_yolo.zip
```

then upload `idd_yolo.zip` to Google Drive.

### What the converter already decided for you

**Front cameras only.** IDD was shot with seven cameras: 41,857 annotated
frames in total, of which 24,312 are forward-facing (`frontFar`, `frontNear`,
`highquality_16k`). NOVA sees one camera, pointing forward. A car photographed
from a 90-degree side camera shares almost no appearance with the same car
ahead of a dashcam, and a nano model has no capacity to waste on a viewpoint
it will never be shown.

**IDD's own train/val split.** Checked on disk: 438 train sequences and 119
val sequences, sharing **zero** sequences. So the official split is already
leak-free, and using it makes this val mAP comparable to published IDD
numbers instead of to a private split nobody can reproduce. This matters -
IDD frames come from continuous drives, so a naive per-frame random split
puts near-identical neighbouring frames on both sides and returns a
flattering, meaningless mAP.

### Why this dataset is the point

Instance counts across the full annotation set:

| class | instances | in COCO? |
|---|---|---|
| motorcycle | 103,608 | yes, but rare in Western street scenes |
| rider | 97,626 | **no** - COCO has no rider class |
| car | 90,520 | yes |
| person | 88,397 | yes |
| **autorickshaw** | **32,280** | **no - not in any Western dataset** |
| truck | 27,837 | yes |
| vehicle fallback | 21,081 | **no** - carts, tractors, oddities |
| bus | 18,745 | yes |
| **animal** | **6,224** | **no on-road equivalent** |

Autorickshaw, rider, vehicle fallback and animal are four classes a
COCO-pretrained model cannot produce at all. They are the answer to
"why not just use an off-the-shelf detector?"

**Set the runtime to GPU first:** Runtime -> Change runtime type -> T4 GPU.
Training on CPU will not finish before your hackathon.

## 1. Check the GPU

If this prints `CPU`, stop and change the runtime type.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv


## 2. Install ultralytics

Pinned. An unpinned install has broken mid-hackathon before — a minor
release changed a default and the run silently trained on the wrong image size.

In [ ]:
!pip install -q ultralytics==8.3.0
import ultralytics; ultralytics.checks()


## 3. Mount Drive and get the dataset onto local disk

Works whether you uploaded `idd_yolo.zip` or the unzipped `idd_yolo` folder.
It looks for either, at the top of My Drive or one or two folders down.

Everything ends up on **local Colab disk**, never trained from Drive:
training reads every image several times per epoch and Drive is a network
mount, so training off it is roughly ten times slower. That is the usual
reason a Colab fine-tune "takes all night".

In [ ]:
from google.colab import drive
from pathlib import Path
import glob, os, shutil, subprocess

drive.mount('/content/drive')
D = '/content/drive/MyDrive'
os.makedirs('/content/data', exist_ok=True)

# Explicit depths rather than a recursive walk: rglob over a large Drive can
# take minutes on the FUSE mount before it finds anything.
zips = sum((glob.glob(p) for p in
            (f'{D}/idd_yolo.zip', f'{D}/*/idd_yolo.zip',
             f'{D}/*/*/idd_yolo.zip')), [])
yamls = sum((glob.glob(p) for p in
             (f'{D}/idd_yolo/data.yaml', f'{D}/*/data.yaml',
              f'{D}/*/*/data.yaml', f'{D}/*/*/*/data.yaml')), [])

if zips:
    print('found zip:', zips[0], f'({os.path.getsize(zips[0])/1e6:.0f} MB)')
    # NOT -q: a quiet unzip that fails sends you debugging the wrong cell.
    r = subprocess.run(['unzip', '-o', zips[0], '-d', '/content/data'],
                       capture_output=True, text=True)
    assert r.returncode == 0, r.stderr[-1000:]
elif yamls:
    src = os.path.dirname(yamls[0])
    print('found folder:', src)
    print('copying to local disk - a few minutes for 12k files...')
    shutil.copytree(src, '/content/data/idd_yolo', dirs_exist_ok=True)
else:
    raise SystemExit(
        'Found neither idd_yolo.zip nor an idd_yolo folder.\n'
        f'Top of your Drive: {sorted(os.listdir(D))[:25]}\n'
        'Upload C:\\NOVA\\idd_yolo.zip to the top of My Drive and re-run.')

got = glob.glob('/content/data/**/data.yaml', recursive=True)
assert got, 'copy/unzip reported success but no data.yaml landed'
ROOT = Path(os.path.dirname(got[0]))
n_train = len(glob.glob(f'{ROOT}/images/train/*.jpg'))
n_val = len(glob.glob(f'{ROOT}/images/val/*.jpg'))
print(f'\nROOT = {ROOT}')
print(f'{n_train} train images, {n_val} val images')
assert n_train and n_val, 'images missing - the upload is incomplete'

## 4. Sanity-check the dataset before training

Two minutes here saves an hour of training something wrong. This confirms
the images and labels line up, and prints the class balance — if a class has
almost no instances, the model will never learn it and you should know that
*before* you present a confusion matrix.

In [ ]:
from pathlib import Path
import collections, yaml

ROOT = Path('/content/data/idd_yolo')
if not (ROOT / 'data.yaml').exists():
    cands = list(Path('/content/data').rglob('data.yaml'))
    assert cands, 'no data.yaml found - check the zip contents'
    ROOT = cands[0].parent
print('dataset root:', ROOT)

cfg = yaml.safe_load((ROOT / 'data.yaml').read_text())
names = cfg['names']
print('classes:', names)

for split in ('train', 'val'):
    imgs = sorted((ROOT / 'images' / split).glob('*.jpg'))
    lbls = sorted((ROOT / 'labels' / split).glob('*.txt'))
    print(f'{split}: {len(imgs)} images, {len(lbls)} labels')
    assert len(imgs) == len(lbls), 'image/label count mismatch'

counts = collections.Counter()
for t in (ROOT / 'labels' / 'train').glob('*.txt'):
    for line in t.read_text().splitlines():
        if line.strip():
            counts[int(line.split()[0])] += 1
print('\ninstances per class in train:')
for i, n in sorted(names.items()):
    print(f'  {n:<18} {counts.get(i, 0)}')


## 5. Look at a few labelled images

Do not skip this. A silent coordinate bug in the converter produces boxes
that are plausible numbers and nonsense on the image, and the training loss
will happily go down anyway.

In [ ]:
import cv2, random, matplotlib.pyplot as plt

imgs = list((ROOT / 'images' / 'train').glob('*.jpg'))
random.seed(0)
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
for ax, p in zip(axes, random.sample(imgs, 3)):
    im = cv2.imread(str(p))[:, :, ::-1].copy()
    h, w = im.shape[:2]
    lab = ROOT / 'labels' / 'train' / (p.stem + '.txt')
    for line in lab.read_text().splitlines():
        c, cx, cy, bw, bh = line.split()
        cx, cy, bw, bh = float(cx)*w, float(cy)*h, float(bw)*w, float(bh)*h
        x1, y1 = int(cx - bw/2), int(cy - bh/2)
        x2, y2 = int(cx + bw/2), int(cy + bh/2)
        cv2.rectangle(im, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(im, names[int(c)], (x1, max(12, y1-4)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
    ax.imshow(im); ax.axis('off')
plt.tight_layout(); plt.show()


## 6. Train

**`yolov8n`, not `s` or `m`.** NOVA's inference budget is roughly 12 ms per
frame on a 6 GB laptop card. `yolov8s` is about 2.5x the cost. A slightly
less accurate detector that runs is worth more than an accurate one that
stalls the loop.

Starting from the COCO checkpoint rather than scratch: COCO already knows
car, bus, truck, motorcycle, bicycle and person. The fine-tune is mostly
teaching it **autorickshaw, rider, animal and vehicle fallback**, plus Indian
street appearance - which is exactly the claim you are making on stage.

Roughly 50-60 min for 40 epochs on a T4 at 10k images. Colab disconnects an
idle tab, so leave it open - and run the checkpoint cell below, which copies
the run to Drive so a disconnect costs you minutes rather than the session.

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
results = model.train(
    data=str(ROOT / 'data.yaml'),
    epochs=40,
    imgsz=640,
    batch=32,
    device=0,
    workers=2,
    patience=10,          # stop early if val mAP plateaus
    project='/content/runs',
    name='nova_idd',
    # Indian road scenes are dense and cluttered; the default 300 detections
    # per image is plenty, but the default mosaic augmentation helps a lot
    # with the small, overlapping two-wheelers that dominate this dataset.
    mosaic=1.0,
    close_mosaic=10,      # turn it off for the last 10 epochs to settle
)


## 6b. Disconnect insurance

Free Colab will drop you, usually at the worst moment. Ultralytics writes a
`last.pt` every epoch, so a dropped session only costs the epoch in progress -
**but only if the weights are somewhere that survives the VM.** Run this now;
re-run it any time you want a fresh copy.

If you do get disconnected: re-run the install and mount cells, then the
**resume** lines at the bottom of this cell instead of the train cell.

In [ ]:
import os
import shutil

os.makedirs('/content/drive/MyDrive/nova_runs', exist_ok=True)
for w in ('best.pt', 'last.pt'):
    src = f'/content/runs/nova_idd/weights/{w}'
    if os.path.exists(src):
        shutil.copy(src, f'/content/drive/MyDrive/nova_runs/{w}')
        print('saved', w, round(os.path.getsize(src) / 1e6, 1), 'MB')
    else:
        print('not there yet:', w)

# --- RESUME after a disconnect: uncomment and run INSTEAD of the train cell.
# Ultralytics reads the epoch count and the optimiser state out of last.pt,
# so it picks up where it stopped rather than starting over.
# shutil.copy('/content/drive/MyDrive/nova_runs/last.pt', '/content/last.pt')
# from ultralytics import YOLO
# YOLO('/content/last.pt').train(resume=True)

## 7. Validate and read the per-class numbers

The per-class table is what you put in your report. Overall mAP hides the
thing judges will ask about — whether it actually detects autorickshaws.

In [ ]:
best = '/content/runs/nova_idd/weights/best.pt'
m = YOLO(best)
metrics = m.val(data=str(ROOT / 'data.yaml'), device=0)
print('mAP50-95:', round(float(metrics.box.map), 4))
print('mAP50   :', round(float(metrics.box.map50), 4))


## 8. Confirm the class names, then save `best.pt`

**This cell is the one that matters for integration.** NOVA does
`names[cls].lower()` and looks the string up in `NAME_TO_CLASS`. If a name
here is missing from that dict, NOVA detects the object and then silently
throws it away — no error, the agent just never appears in the risk map.

In [ ]:
m = YOLO(best)
NOVA_KNOWS = {'car','truck','bus','motorcycle','bicycle','autorickshaw',
              'auto-rickshaw','rider','motorcyle','animal','cart','pushcart',
              'vehicle fallback','caravan','trailer','person','dog','cow','horse'}
for i, n in m.names.items():
    mark = 'ok ' if n.lower() in NOVA_KNOWS else 'NOT MAPPED ->'
    print(f'  {mark} {i}: {n}')

!cp "$best" /content/drive/MyDrive/nova_best.pt
print('\nsaved to Drive as nova_best.pt')
from google.colab import files
files.download(best)


## 9. Where `best.pt` actually gets used

Put `best.pt` in `C:\NOVA\` and run **NOVA 2.0, on real road video**:

```powershell
python scripts\nova_video.py --video road.mp4 --weights best.pt --fov 70 --loop
```

**Not `nova_drive.py --weights best.pt`.** That is the natural thing to try,
and on this machine it cannot work: attaching any camera sensor wedges the
CARLA server about six ticks later, at every resolution down to 64x64 (see
CLAUDE.md for the full list of what was ruled out by measurement). So the
CARLA demo runs on simulator ground truth (`--ground-truth --no-cameras`) and
shows closed-loop driving; the video demo runs this detector and shows
perception. If the camera issue is ever fixed, `nova_drive.py --weights
best.pt` needs no code change - the flag is already wired.

Nothing else changes either way. `VisionPerception` takes the weights path as
an argument, ByteTrack is tracker-side and model-agnostic, and the class names
are already in `NAME_TO_CLASS`. The rest of the pipeline - depth
back-projection, prediction, risk map, planner - never learns which detector
produced the tracks. That is the architecture claim, and this is the test of
it: swapping a COCO model for an IDD model touches one filename.

### Keep the numbers

Run `nova_video.py` twice on the same clip, once with `yolov8n.pt` and once
with `best.pt`, and count tracked agents per frame. The COCO model cannot emit
`autorickshaw` at all, so every auto in the footage is either missed entirely
or misclassified as a car - and a misclassified auto gets a car's footprint
and a car's erraticness prior in the risk map, which changes what the planner
does. **That difference is your result**, and it is a far stronger thing to
show a judge than an mAP number on its own.